# Construcción de aplicaciones de generación de imágenes (Azure OpenAI)

Los LLM no solo sirven para la generación de texto. También puedes generar imágenes a partir de descripciones textuales. Las imágenes como modalidad son útiles en MedTech, arquitectura, turismo, desarrollo de juegos, marketing y más. En esta lección trabajamos con los modelos de **GPT Image** actuales en Microsoft Foundry.

## Objetivos de aprendizaje

Al final de esta lección serás capaz de:

- Explicar qué es la generación de imágenes y dónde es útil.
- Entender la familia de modelos `gpt-image` y cómo difiere de los modelos antiguos DALL·E.
- Construir una aplicación de generación de imágenes y editar imágenes.

## ¿Qué es la generación de imágenes?

Los modelos de generación de imágenes crean imágenes a partir de un texto de entrada. Los modelos modernos como `gpt-image` aprenden la relación entre texto e imágenes durante el entrenamiento, y luego transforman iterativamente ruido aleatorio en una imagen que coincide con tu solicitud.

Dos familias bien conocidas de modelos de imágenes son:

- **`gpt-image` (OpenAI)** — la generación actual usada en esta lección. Soporta generación de imagen a partir de texto y edición de imágenes (pintura dentro de una máscara).
- **Midjourney** — un modelo de terceros popular con su propio servicio y flujo de trabajo basado en Discord.

> Los modelos antiguos de imágenes de OpenAI — **DALL·E 2** y **DALL·E 3** — son obsoletos. DALL·E 3 ya no está disponible para nuevos despliegues, y funciones como `create_variation` solo existían en DALL·E 2. Usa los modelos `gpt-image` para nuevas aplicaciones.

En Microsoft Foundry, **`gpt-image-2`** es el modelo de imagen más reciente y con mayor capacidad, y es el recomendado por defecto. `gpt-image-1.5` y `gpt-image-1-mini` también están disponibles en general.

> **Importante:** los modelos `gpt-image` devuelven la imagen generada en **base64** (`b64_json`), no como una URL. Tu código decodifica la cadena base64 a bytes y la guarda — no hay URL de imagen para descargar.


## Construyendo tu primera aplicación de generación de imágenes

Entonces, ¿qué se necesita para construir una aplicación de generación de imágenes? Necesitas las siguientes librerías:

- **python-dotenv**, se recomienda mucho usar esta librería para mantener tus secretos en un archivo *.env* separado del código.
- **openai**, esta librería es la que usarás para interactuar con la API de OpenAI.
- **pillow**, para trabajar con imágenes en Python.

Si aún no lo has hecho, sigue las instrucciones en la página de [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) para crear un recurso y modelo de Microsoft Foundry. Selecciona **gpt-image-2** como el modelo (el modelo más reciente de imágenes de Azure OpenAI; DALL·E es legado).

1. Crea un archivo *.env* con el siguiente contenido:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Encuentra esta información en el portal de Microsoft Foundry para tu recurso en la sección "Deployments".



1. Recolecta las bibliotecas anteriores en un archivo llamado *requirements.txt* así:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. A continuación, crea un entorno virtual e instala las bibliotecas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para Windows, usa los siguientes comandos para crear y activar tu entorno virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Agrega el siguiente código en un archivo llamado *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importar dotenv
    dotenv.load_dotenv()

    # configurar el cliente de Azure OpenAI (Microsoft Foundry).
    # Los modelos de imagen necesitan una versión reciente de la API — verifica la documentación de Foundry para la que requiere tu modelo.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Crear una imagen usando la API de generación de imágenes
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ingresa tu texto de prompt aquí
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # p.ej. "gpt-image-2"
        )
        # Establecer el directorio para la imagen almacenada
        image_dir = os.path.join(os.curdir, 'images')

        # Si el directorio no existe, créalo
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializar la ruta de la imagen (nota: el tipo de archivo debe ser png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Los modelos gpt-image retornan la imagen como base64 (b64_json), no una URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Mostrar la imagen en el visor de imágenes predeterminado
        image = Image.open(image_path)
        image.show()

    # capturar excepciones
    except BadRequestError as err:
        print(err)

    ```

Vamos a explicar este código:

- Primero, importamos las librerías que necesitamos, incluyendo la librería OpenAI, la librería dotenv, el módulo base64, y la librería Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- A continuación, cargamos las variables de entorno desde el archivo *.env*.

    ```python
    # importar dotenv
    dotenv.load_dotenv()
    ```

- Después, configuramos el cliente Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Luego, generamos la imagen:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduzca su texto de solicitud aquí
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Los modelos `gpt-image` retornan la imagen como un string **base64** en `data[0].b64_json`. La decodificamos a bytes y la escribimos en un archivo — no hay una URL para descargarla.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Por último, abrimos la imagen y usamos el visor de imágenes estándar para mostrarla:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Más detalles sobre la generación de la imagen

Veamos los parámetros de `images.generate`:

- **prompt**, es el texto guía usado para generar la imagen. Aquí es "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, es el tamaño de la imagen generada (por ejemplo `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, es la cantidad de imágenes generadas. Aquí generamos una.
- **model**, es el nombre de tu implementación del modelo de imagen (por ejemplo `gpt-image-2`).

> Los modelos de imagen no aceptan un parámetro `temperature` — eso es un control para generación de texto. Para obtener variedad, llama a la API nuevamente; para reducir variedad, haz tu prompt más específico.

## Capacidades adicionales de generación de imagen

Has visto cómo generar una imagen con unas pocas líneas de Python. Los modelos `gpt-image` también pueden **editar** una imagen existente. Proporcionando una imagen, una **máscara** opcional (que marca el área a cambiar), y un prompt, puedes alterar parte de una imagen — por ejemplo, añadir un sombrero a nuestro conejo.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# las ediciones también se devuelven como base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

La imagen base contiene solo el conejo; la imagen final tiene el sombrero puesto en el conejo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
